# 📍 18 · Colocalization and Spatial Statistics
### Are two things together more than chance would allow? From pixel intensities to point patterns, the same question, the same two ingredients: a measurement and a null distribution

**Part I · Quantifying and modern architectures** · [⟵ 17 Restoration & inverse problems](17_restoration_and_inverse_problems.ipynb) · [🏠 Course home](../README.md) · [19 Attention, transformers & diffusion ⟶](19_attention_transformers_diffusion.ipynb)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/18_colocalization_and_spatial_statistics.ipynb)

| | |
|---|---|
| **Hardware target** | CPU; every computation in this notebook runs in well under a second |
| **Core frameworks** | NumPy, SciPy (spatial), Matplotlib |
| **Prerequisites** | Module 02 (digital images), Module 03 (thresholding), Module 10/11 (segmentation) |
| **Estimated time** | 70-90 minutes |
| **02-740 topics covered** | quantitative colocalization and spatial-statistics methods from the course materials: pixel-based (Pearson) and area-based (Manders') colocalization, Costes automatic thresholding and randomization significance testing, nearest-neighbour analysis (within- and cross-population), the Clark-Evans index, Ripley's L function, null-distribution taxonomy, Delaunay/Voronoi spatial graphs |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All images, channels and point patterns in this notebook are SIMULATED by code — nothing is downloaded.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- explain why raw coordinates or pixel intensities are not yet "data" — and why every colocalization or spatial-pattern question needs both a numerical measurement and a null distribution to compare it against;
- compute Pearson's correlation between two channels and state when it is, and is not, a meaningful colocalization measure;
- compute Manders' overlap coefficients, explain why naive thresholds make them uninformative, and use Costes' method to set the threshold automatically and to test whether an observed correlation is likely due to chance;
- compute nearest-neighbour distances (within one population and across two), the Clark-Evans index, and Ripley's L function against a Monte Carlo CSR null — and explain what Ripley's L reveals about *scale* that a single global index cannot;
- choose an appropriate null distribution for a spatial question from a short taxonomy (CSR, label randomization, toroidal shift, block/chunk shuffling), and state clearly what colocalization and spatial-pattern analysis can, and cannot, establish about biology.

## 🗺️ Contents

1. The big picture: a measurement, and something to compare it against
2. Pixel-based colocalization: Pearson's correlation
3. Area-based colocalization: Manders' coefficients and the threshold problem
4. Costes' method: automatic thresholding and a randomization significance test
5. From pixels to objects: spatial point patterns and nearest-neighbour distances
6. Clark-Evans index and Ripley's L function against a CSR null
7. Choosing a null distribution, spatial graphs, and what spatial analysis can (and cannot) tell you

## 🧫 The biological question

Two fluorescently-labelled proteins look like they occupy "the same place" in a merged microscopy image — do they actually colocalize more than you'd expect by chance, and what would that even mean, precisely? The same question, reframed, applies just as well to whether two cell types cluster together in a tissue section, or whether a population of detected objects is clustered, dispersed, or indistinguishable from pure randomness. Both questions need exactly the same two ingredients: a way to turn raw positions or intensities into one comparable number, and a null distribution representing "no real relationship" to compare that number against. This notebook builds both halves — for continuous pixel intensities (colocalization) and for discrete point coordinates (spatial statistics) — and is explicit throughout about what the resulting numbers can, and cannot, tell you about the underlying biology.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.spatial import Delaunay, Voronoi
SEED = 18
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

We reuse `simulate_two_channel_image`, `pearson_colocalization`, `manders_coefficients`, `simulate_point_pattern`, `nearest_neighbor_distances`, `clark_evans_index`, `ripley_l`, `csr_envelope`, and `show_image_grid`.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def simulate_two_channel_image(size: int = 160, n_spots: int = 60, association: float = 0.7,
                                psf_sigma: float = 1.3, seed: int = 0):
    """Two SIMULATED fluorescence channels (protein A, protein B) with a known association.

    ``association`` in [0, 1] is the fraction of channel-B spots placed near an existing
    channel-A spot (true colocalization); the rest are placed independently at random. Returns
    ``chA``, ``chB`` (size x size images), ``points_a``, ``points_b`` (coordinates) and
    ``true_association`` (the parameter used, for the ground-truth check).
    """
    rng = np.random.default_rng(seed)
    margin = size * 0.08
    pts_a = rng.uniform(margin, size - margin, size=(n_spots, 2))
    pts_b = []
    for i in range(n_spots):
        if rng.random() < association:
            jitter = rng.normal(0, size * 0.015, size=2)
            pts_b.append(np.clip(pts_a[i] + jitter, margin, size - margin))
        else:
            pts_b.append(rng.uniform(margin, size - margin, size=2))
    pts_b = np.array(pts_b)

    def _render(pts):
        canvas = np.zeros((size, size))
        for cy, cx in pts:
            _draw_blob(canvas, cy, cx, size * 0.018, "round", rng)
        canvas = blur_image(canvas, gaussian_psf(psf_sigma))
        return add_gaussian_noise(canvas, 0.02, rng).clip(0, None)

    chA, chB = _render(pts_a), _render(pts_b)
    return dict(chA=chA, chB=chB, points_a=pts_a, points_b=pts_b,
                true_association=association, source="SIMULATED")


def pearson_colocalization(ch_a: np.ndarray, ch_b: np.ndarray, mask: np.ndarray | None = None) -> float:
    """Pearson's colocalization coefficient: the ordinary linear correlation between two
    channels' pixel intensities, restricted to ``mask`` if given (e.g. just the cell interior)."""
    a = ch_a[mask] if mask is not None else ch_a.ravel()
    b = ch_b[mask] if mask is not None else ch_b.ravel()
    a, b = a - a.mean(), b - b.mean()
    denom = np.sqrt((a ** 2).sum() * (b ** 2).sum())
    return float((a * b).sum() / denom) if denom > 0 else 0.0


def manders_coefficients(ch_a: np.ndarray, ch_b: np.ndarray, thr_a: float, thr_b: float):
    """Manders' colocalization coefficients M1, M2 at the given intensity thresholds: the
    fraction of channel A's (resp. B's) total signal that sits in pixels where the OTHER
    channel is also above its threshold."""
    pos_b = ch_b > thr_b
    pos_a = ch_a > thr_a
    m1 = ch_a[pos_b].sum() / (ch_a.sum() + 1e-9)
    m2 = ch_b[pos_a].sum() / (ch_b.sum() + 1e-9)
    return float(m1), float(m2)


def simulate_point_pattern(n_points: int = 150, size: float = 100.0,
                            pattern: str = "random", cluster_std: float = 4.0,
                            n_clusters: int = 8, min_dist: float = 0.0,
                            seed: int = 0) -> np.ndarray:
    """SIMULATED 2-D point coordinates with a known spatial ``pattern``.

    ``pattern`` is one of ``"random"`` (complete spatial randomness), ``"clustered"``
    (Gaussian clumps around ``n_clusters`` parents) or ``"dispersed"`` (points repel each
    other, enforcing roughly ``min_dist``). Returns an (n_points, 2) array; the pattern name
    is the ground truth used in the 🧪 checks.
    """
    rng = np.random.default_rng(seed)
    if pattern == "random":
        pts = rng.uniform(0, size, size=(n_points, 2))
    elif pattern == "clustered":
        centers = rng.uniform(size * 0.15, size * 0.85, size=(n_clusters, 2))
        choice = rng.integers(0, n_clusters, size=n_points)
        pts = centers[choice] + rng.normal(0, cluster_std, size=(n_points, 2))
        pts = np.clip(pts, 0, size)
    elif pattern == "dispersed":
        min_dist = min_dist or size / np.sqrt(n_points) * 0.9
        pts = []
        attempts = 0
        while len(pts) < n_points and attempts < n_points * 200:
            attempts += 1
            cand = rng.uniform(0, size, size=2)
            if all(np.hypot(*(cand - p)) > min_dist for p in pts):
                pts.append(cand)
        pts = np.array(pts)
    else:
        raise ValueError(f"unknown pattern {pattern!r}")
    return pts


def nearest_neighbor_distances(points: np.ndarray, other: np.ndarray | None = None) -> np.ndarray:
    """For each point, the distance to its nearest neighbour (in ``other`` if given — a CROSS
    nearest-neighbour distance between two populations — else in ``points`` itself)."""
    target = points if other is None else other
    d = np.sqrt(((points[:, None, :] - target[None, :, :]) ** 2).sum(axis=2))
    if other is None:
        np.fill_diagonal(d, np.inf)
    return d.min(axis=1)


def clark_evans_index(points: np.ndarray, area: float) -> float:
    """The Clark–Evans index: observed mean nearest-neighbour distance divided by the distance
    expected under complete spatial randomness (CSR) at the same point density. CE < 1 means
    clustering, CE ≈ 1 random, CE > 1 dispersion/regularity."""
    n = len(points)
    observed = nearest_neighbor_distances(points).mean()
    density = n / area
    expected = 1.0 / (2 * np.sqrt(density))
    return float(observed / expected)


def ripley_l(points: np.ndarray, radii: np.ndarray, area: float) -> np.ndarray:
    """Ripley's L function L(r) − r at each radius in ``radii``: positive means more
    neighbours within r than expected under CSR (clustering at that scale), negative means
    fewer (dispersion); edge correction is omitted for simplicity (use a border guard band)."""
    n = len(points)
    d = np.sqrt(((points[:, None, :] - points[None, :, :]) ** 2).sum(axis=2))
    np.fill_diagonal(d, np.inf)
    density = n / area
    l_vals = []
    for r in radii:
        counts = (d <= r).sum(axis=1).mean()
        k_r = counts / density
        l_vals.append(np.sqrt(k_r / np.pi) - r)
    return np.array(l_vals)


def csr_envelope(n_points: int, size: float, radii: np.ndarray, n_sims: int = 20, seed: int = 0):
    """Monte Carlo null distribution (complete spatial randomness) for Ripley's L: simulate
    ``n_sims`` random point sets of the same size/density and return the (low, high) envelope
    of L(r) − r across simulations, for comparison with an observed curve."""
    rng = np.random.default_rng(seed)
    curves = []
    for s in range(n_sims):
        pts = rng.uniform(0, size, size=(n_points, 2))
        curves.append(ripley_l(pts, radii, size ** 2))
    curves = np.array(curves)
    return curves.min(axis=0), curves.max(axis=0)


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes

---

## 1 · The big picture: a measurement, and something to compare it against

Coordinates and pixel intensities are not, by themselves, data about *relationships* — "cells are clustered," "proteins colocalize," and "cell density is high" are all statements that only mean something **relative to an expectation**. Answering any of them takes the same pipeline every time: turn coordinates or intensities into a measurement, then compare that measurement against a **null distribution** describing what you would see if there were no real relationship at all.

Which measurement to use depends on one practical question first: **can you segment discrete objects** in your data? If yes, you have point coordinates, and Section 5 onward (nearest-neighbour distances, Clark-Evans, Ripley's L) applies. If no — you only have continuous pixel intensities across two channels — Sections 2 to 4 (Pearson, Manders, Costes) apply instead. Within pixel-based analysis, a second question matters: do the two channels occupy **roughly similar areas** of the image? If so, a single global Pearson correlation is directly interpretable; if one channel is much sparser or more punctate than the other (e.g. a small organelle marker against a diffuse cytoplasmic stain), correlation is harder to interpret and threshold-based Manders' coefficients are the better tool.

### 📘 Lecture notes — a good null distribution is not just "anything random"

A useful null distribution has to do two things at once: reflect the aspects of the observed data that are *not* in question (e.g. the same number of objects, the same overall density), while genuinely differing in the one aspect that *is* in question (e.g. the actual spatial arrangement or channel alignment). Complete spatial randomness (CSR) — keep the same number of objects, randomize their positions — is the simplest and most common choice, and Monte Carlo sampling (repeating CSR many times rather than relying on one random draw) turns it into a usable envelope or p-value rather than a single noisy comparison. Section 7 returns to this with a fuller taxonomy of null distributions, because CSR is not always the right choice.

## 2 · Pixel-based colocalization: Pearson's correlation

The most direct pixel-based colocalization measure is just the ordinary linear correlation coefficient between two channels' intensities at every pixel: "Pearson's correlation reports on whether two signals co-vary across a region." It ranges from $-1$ (anti-correlated) through $0$ (no linear relationship) to $+1$ (perfectly correlated), and — like any correlation — it is most directly interpretable when the two channels occupy comparable areas and intensity ranges; computing it over mismatched regions (e.g. whole-cell vs. a sub-compartment where only one channel is expressed) can give misleading values, which is why region-of-interest choice matters as much as the formula itself.

In [ ]:
# ── 2.1 Two simulated two-channel scenes: strongly associated vs. near-independent ──
high_assoc = simulate_two_channel_image(size=160, n_spots=60, association=0.95, seed=SEED)
near_indep = simulate_two_channel_image(size=160, n_spots=60, association=0.0, seed=SEED)

def merge_rgb(scene):
    zeros = np.zeros_like(scene["chA"])
    return np.dstack([scene["chA"], scene["chB"], zeros]) / max(scene["chA"].max(), scene["chB"].max())

show_image_grid([high_assoc["chA"], high_assoc["chB"], merge_rgb(high_assoc),
                  near_indep["chA"], near_indep["chB"], merge_rgb(near_indep)],
                 titles=["A (high assoc.)", "B (high assoc.)", "merge (high assoc.)",
                         "A (near-independent)", "B (near-independent)", "merge (near-independent)"],
                 ncols=3)

🔍 **What just happened?** In the merged images (channel A in red, channel B in green), strongly associated spots turn yellow where the two channels overlap; near-independent spots stay mostly separate red and green dots. This is the qualitative version of what Pearson's correlation now quantifies.

In [ ]:
# ── 2.2 Pearson's correlation, and the cytofluorogram scatter plot ──
r_high = pearson_colocalization(high_assoc["chA"], high_assoc["chB"])
r_low = pearson_colocalization(near_indep["chA"], near_indep["chB"])

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].scatter(high_assoc["chA"].ravel(), high_assoc["chB"].ravel(), s=2, alpha=0.2)
axes[0].set_title(f"high association\nPearson r = {r_high:.3f}")
axes[1].scatter(near_indep["chA"].ravel(), near_indep["chB"].ravel(), s=2, alpha=0.2)
axes[1].set_title(f"near-independent\nPearson r = {r_low:.3f}")
for ax in axes:
    ax.set_xlabel("channel A intensity")
    ax.set_ylabel("channel B intensity")
plt.tight_layout()
plt.show()
print(f"high association:    Pearson r = {r_high:.3f}")
print(f"near-independent:     Pearson r = {r_low:.3f}")

🔍 **What just happened?** This scatter of every pixel's channel-A intensity against its channel-B intensity is a **cytofluorogram** — the standard way to visualise pixel-based colocalization. A tight diagonal cloud means strong positive correlation; a shapeless blob near the origin means little to no linear relationship. Notice that the "near-independent" scene's Pearson $r$ is small but **not exactly zero** — Section 4 comes back to exactly this point.

✏️ **Your turn.** Re-run cell 2.1 with `association=0.5` for both scenes. Where does the resulting Pearson $r$ fall relative to the two values just printed? Is the relationship between `association` (the ground-truth simulation parameter) and the measured Pearson $r$ linear, or does it look like something else?

## 3 · Area-based colocalization: Manders' coefficients and the threshold problem

Manders' coefficients answer a different, complementary question from Pearson's: "for a specified threshold, how many pixels are positive for both signals, as a fraction of either signal" — giving two numbers, $M_1$ (the fraction of channel A's signal that overlaps channel B) and $M_2$ (the reverse). Unlike Pearson's correlation, Manders' coefficients do not require the two channels' intensities to be linearly related at all — only that you can decide, pixel by pixel, whether each channel is "on" or "off." That requires a **threshold**, and the choice of threshold is not a technicality — it is the whole ballgame.

In [ ]:
# ── 3.1 A naive near-zero threshold: are Manders' coefficients still informative? ──
m1_naive_high, m2_naive_high = manders_coefficients(high_assoc["chA"], high_assoc["chB"], thr_a=1e-6, thr_b=1e-6)
m1_naive_low, m2_naive_low = manders_coefficients(near_indep["chA"], near_indep["chB"], thr_a=1e-6, thr_b=1e-6)
print(f"high association,   near-zero threshold:  M1 = {m1_naive_high:.3f}  M2 = {m2_naive_high:.3f}")
print(f"near-independent,    near-zero threshold:  M1 = {m1_naive_low:.3f}  M2 = {m2_naive_low:.3f}")

🔍 **What just happened?** With almost no threshold at all, essentially every nonzero pixel counts as "signal" in both channels — so $M_1$/$M_2$ end up fairly high for *both* scenes, and the gap between them (the only thing that is actually informative about real association) is much smaller than the underlying difference in association. On the whole image, with no meaningful threshold applied, Manders' coefficients are almost always close to 1 — which is why a threshold must be defined deliberately, not left at an arbitrary near-zero default.

### 📘 Lecture notes — Manders' coefficients are a global analysis, with real limitations

Manders' coefficients summarise the whole image (or region) into two numbers, so, like Pearson's correlation, they say nothing about *where* overlap occurs within that region. They also depend entirely on the threshold choice, which traditionally required additional "mono-staining" control images (imaging each fluorophore completely alone, without the other, to calibrate what counts as real signal versus background) — an extra experiment that is not always practical. And because Manders' coefficients are a single global summary, they are generally more informative compared *across conditions* (e.g. wild-type vs. knockout, or one protein against two different partners) than interpreted in isolation. Section 4 gives a way to set the threshold directly from the data, without a separate control image.

✏️ **Your turn.** Try a few manual thresholds using `np.percentile(high_assoc["chA"], pct)` for `pct` in `{50, 90, 99}` (and the matching percentile of `chB`) as `thr_a`/`thr_b`. Does a higher percentile threshold increase or decrease the *gap* between the high-association and near-independent scenes' $M_1$ values, compared to the near-zero threshold above?

## 4 · Costes' method: automatic thresholding and a randomization significance test

Costes et al. (2004) addressed the threshold problem in two complementary ways, both reused below: an **automatic threshold**, derived from the data itself rather than a separate control image, and a **randomization test**, which asks whether the whole-image Pearson correlation is actually larger than chance would produce — a validation tool to show that an observed correlation is not just noise.

In [ ]:
# ── 4.1 Costes' automatic threshold ──
def costes_threshold(ch_a, ch_b, n_steps=200):
    """Scan a threshold down along the regression line of B on A; stop at the highest
    threshold where the BELOW-threshold pixel population has zero (or negative) Pearson
    correlation — i.e. the point past which sub-threshold pixels look like uncorrelated
    background rather than real signal."""
    a_flat, b_flat = ch_a.ravel(), ch_b.ravel()
    slope, intercept = np.polyfit(a_flat, b_flat, 1)
    t_candidates = np.linspace(a_flat.max(), a_flat.min(), n_steps)
    chosen_ta, chosen_tb = t_candidates[0], slope * t_candidates[0] + intercept
    for ta in t_candidates:
        tb = slope * ta + intercept
        below = (a_flat < ta) & (b_flat < tb)
        if below.sum() < 50:
            continue
        a_b, b_b = a_flat[below], b_flat[below]
        a_c, b_c = a_b - a_b.mean(), b_b - b_b.mean()
        denom = np.sqrt((a_c ** 2).sum() * (b_c ** 2).sum())
        r_below = (a_c * b_c).sum() / denom if denom > 0 else 0.0
        chosen_ta, chosen_tb = ta, tb
        if r_below <= 0:
            break
    return chosen_ta, chosen_tb

ta_high, tb_high = costes_threshold(high_assoc["chA"], high_assoc["chB"])
ta_low, tb_low = costes_threshold(near_indep["chA"], near_indep["chB"])
m1_costes_high, m2_costes_high = manders_coefficients(high_assoc["chA"], high_assoc["chB"], ta_high, tb_high)
m1_costes_low, m2_costes_low = manders_coefficients(near_indep["chA"], near_indep["chB"], ta_low, tb_low)

print(f"high association:  Costes threshold A={ta_high:.4f}  ->  M1={m1_costes_high:.3f}  M2={m2_costes_high:.3f}")
print(f"near-independent:   Costes threshold A={ta_low:.4f}  ->  M1={m1_costes_low:.3f}  M2={m2_costes_low:.3f}")

🧪 **Ground-truth check.** Compare these $M_1$ values against Section 3's near-zero-threshold values. The *gap* between the high-association and near-independent scenes should be noticeably larger here — the automatic threshold is doing real work, not just picking an arbitrary cutoff.

In [ ]:
# ── 4.2 Costes' randomization: is the observed Pearson r bigger than chance alone would give? ──
def costes_randomization_test(ch_a, ch_b, block_size=8, n_shuffles=200, seed=0):
    """Shuffle channel B in BLOCKS (not independent pixels) and recompute Pearson r each
    time, to build a null distribution. Block shuffling (rather than per-pixel shuffling)
    respects the sub-resolution pixel coupling created by the point-spread function —
    pixels within one PSF-width of each other are not statistically independent, so a
    block size close to the PSF width is the honest choice."""
    rng = np.random.default_rng(seed)
    observed_r = pearson_colocalization(ch_a, ch_b)
    h, w = ch_b.shape
    bh, bw = h // block_size, w // block_size
    cropped = ch_b[:bh * block_size, :bw * block_size]
    blocks = cropped.reshape(bh, block_size, bw, block_size).swapaxes(1, 2).reshape(bh * bw, block_size, block_size)
    null_rs = np.empty(n_shuffles)
    for i in range(n_shuffles):
        order = rng.permutation(bh * bw)
        shuffled_blocks = blocks[order].reshape(bh, bw, block_size, block_size).swapaxes(1, 2)
        shuffled = ch_b.copy()
        shuffled[:bh * block_size, :bw * block_size] = shuffled_blocks.reshape(bh * block_size, bw * block_size)
        null_rs[i] = pearson_colocalization(ch_a, shuffled)
    return observed_r, null_rs

obs_high, null_high = costes_randomization_test(high_assoc["chA"], high_assoc["chB"], seed=SEED)
obs_low, null_low = costes_randomization_test(near_indep["chA"], near_indep["chB"], seed=SEED)
p_high = (null_high >= obs_high).mean()
p_low = (null_low >= obs_low).mean()

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].hist(null_high, bins=25, color="gray")
axes[0].axvline(obs_high, color="crimson", linewidth=2)
axes[0].set_title(f"high association\nobserved r={obs_high:.3f}  p={p_high:.3f}")
axes[1].hist(null_low, bins=25, color="gray")
axes[1].axvline(obs_low, color="crimson", linewidth=2)
axes[1].set_title(f"near-independent\nobserved r={obs_low:.3f}  p={p_low:.3f}")
for ax in axes:
    ax.set_xlabel("Pearson r under block-shuffled null")
plt.tight_layout()
plt.show()
print(f"high association:  observed r={obs_high:.3f}  null mean={null_high.mean():.3f}  gap={obs_high - null_high.mean():.3f}  p={p_high:.3f}")
print(f"near-independent:   observed r={obs_low:.3f}  null mean={null_low.mean():.3f}  gap={obs_low - null_low.mean():.3f}  p={p_low:.3f}")

🔍 **What just happened?** For the high-association scene, the observed $r$ sits far outside the entire null histogram — a large gap, overwhelming statistical significance, exactly the "good validation tool, to show that it's not random" the randomization test is meant to be. For the near-independent scene, the observed $r$ is small and sits only barely past the edge of the null distribution — technically still "significant" at a conventional threshold, but a small, marginal effect rather than a convincing one. **Statistical significance and effect size are not the same thing**: a p-value tells you whether an effect is probably real, not whether it is large or biologically important — both numbers matter, and reporting only one of them is misleading.

✏️ **Your turn.** Re-run cell 4.2 with `block_size=1` (pixel-by-pixel shuffling, ignoring the point-spread function's local correlation entirely) instead of `8`. Does the near-independent scene's p-value get smaller or larger? Given the taxonomy in Section 7, which null is more trustworthy here, and why?

## 5 · From pixels to objects: spatial point patterns and nearest-neighbour distances

Once objects are segmented into discrete coordinates, a different family of questions opens up: not "do these two continuous signals correlate," but "how are these discrete objects arranged in space, and how does one population of objects relate to another?" The most basic spatial feature of a point pattern is simply **distance to nearest neighbour** — computed either within a single population, or, just as usefully, **across** two populations (e.g. "how far is each cell of type A from its nearest cell of type B?"). That cross-population question is directional: A→B distances are generally *not* the same as B→A distances, especially when the two populations have different sizes or densities.

In [ ]:
# ── 5.1 Within-population and cross-population nearest-neighbour distances ──
FIELD = 100.0
pts_a = simulate_point_pattern(n_points=40, size=FIELD, pattern="clustered", cluster_std=6.0, n_clusters=5, seed=SEED)
pts_b = simulate_point_pattern(n_points=40, size=FIELD, pattern="clustered", cluster_std=6.0, n_clusters=5, seed=SEED + 1)

within_a = nearest_neighbor_distances(pts_a)
a_to_b = nearest_neighbor_distances(pts_a, other=pts_b)
b_to_a = nearest_neighbor_distances(pts_b, other=pts_a)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(pts_a[:, 1], pts_a[:, 0], c="crimson", label="population A")
ax.scatter(pts_b[:, 1], pts_b[:, 0], c="steelblue", label="population B")
ax.set_title("Two independently-clustered populations")
ax.legend()
plt.show()

print(f"within-A mean NN distance:   {within_a.mean():.2f}")
print(f"A -> B mean cross NN distance: {a_to_b.mean():.2f}")
print(f"B -> A mean cross NN distance: {b_to_a.mean():.2f}")

🔍 **What just happened?** Even though populations A and B were generated completely independently of each other here, their within-population nearest-neighbour distance is noticeably smaller than either cross-population distance — because each population is internally clustered (tight within itself), while the two populations' cluster centres happen to fall in different places. Note also that A→B and B→A are close but not identical: cross-distance can depend on each population's own density, so directionality always deserves a second look before concluding anything about which population is "near" the other.

✏️ **Your turn.** Change `pts_b`'s seed to `SEED` (the same seed as `pts_a`, but keep `n_clusters=5`) and regenerate. Since both populations would then share the same five cluster centres, how would you expect the cross-population distances to change relative to the within-population distance? Check your prediction against the printed numbers.

## 6 · Clark-Evans index and Ripley's L function against a CSR null

The **Clark-Evans index** condenses an entire point pattern's nearest-neighbour statistics into one number: the observed mean nearest-neighbour distance, divided by the distance expected under complete spatial randomness (CSR) at the same point density. $CE < 1$ indicates clustering, $CE \approx 1$ indicates a pattern indistinguishable from random, and $CE > 1$ indicates dispersion (regularity).

In [ ]:
# ── 6.1 Clark-Evans index for three qualitatively different patterns ──
for pattern in ["random", "clustered", "dispersed"]:
    pts = simulate_point_pattern(n_points=150, size=FIELD, pattern=pattern, cluster_std=4.0, n_clusters=8, seed=SEED)
    ce = clark_evans_index(pts, area=FIELD ** 2)
    print(f"{pattern:10s}  CE = {ce:.3f}")

🔍 **What just happened?** The three patterns land on the three sides of $CE=1$ exactly as the formula predicts: clustered falls below 1, random sits close to 1, and dispersed (points that actively repel each other, as our `simulate_point_pattern(pattern="dispersed")` enforces) lands above 1. But a single number like this cannot say *at what spatial scale* the clustering or dispersion happens — which is exactly what Ripley's L function adds.

In [ ]:
# ── 6.2 Ripley's L function against a Monte Carlo CSR null ──
radii = np.linspace(2, 25, 12)
fig, ax = plt.subplots(figsize=(7, 5))
for pattern, color in [("random", "gray"), ("clustered", "crimson"), ("dispersed", "steelblue")]:
    pts = simulate_point_pattern(n_points=150, size=FIELD, pattern=pattern, cluster_std=4.0, n_clusters=8, seed=SEED)
    l_vals = ripley_l(pts, radii, area=FIELD ** 2)
    ax.plot(radii, l_vals, "o-", color=color, label=pattern)
lo, hi = csr_envelope(n_points=150, size=FIELD, radii=radii, n_sims=40, seed=SEED)
ax.fill_between(radii, lo, hi, color="black", alpha=0.15, label="CSR envelope (Monte Carlo)")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("radius r")
ax.set_ylabel("L(r) - r")
ax.set_title("Ripley's L function vs. a CSR null envelope")
ax.legend()
plt.show()

🔍 **What just happened?** The "random" curve stays close to the CSR envelope at every radius tested, as it should — it *is* CSR. The "clustered" curve sits clearly above the envelope at every radius shown here: more neighbours within each radius than chance would produce. The "dispersed" curve sits below: fewer close neighbours than chance, at every radius shown — exactly the regularity its generation process (points placed with an enforced minimum separation) should produce.

In [ ]:
# ── 6.3 Ripley's L reveals the *scale* of clustering, not just its presence ──
radii_fine = np.linspace(2, 25, 12)
for cluster_std, label in [(2.0, "tight clusters"), (10.0, "loose clusters")]:
    pts = simulate_point_pattern(n_points=150, size=FIELD, pattern="clustered",
                                  cluster_std=cluster_std, n_clusters=8, seed=SEED)
    l_vals = ripley_l(pts, radii_fine, area=FIELD ** 2)
    peak_r = radii_fine[int(np.argmax(l_vals))]
    print(f"{label:14s} (cluster_std={cluster_std:4.1f}):  peak L(r)-r = {l_vals.max():.2f} at r = {peak_r:.1f}")

🔍 **What just happened?** Both patterns are "clustered" by Clark-Evans' standard — but *where* the clustering signal peaks shifts to a larger radius as the clusters themselves get looser (larger `cluster_std`). This is precisely what a single Clark-Evans number cannot tell you: the Clark-Evans index would call both patterns "clustered" with no information about the characteristic size of that clustering, while Ripley's L recovers it directly from where its curve peaks.

### 📘 Lecture notes — honesty about edge effects

`ripley_l` here deliberately omits an edge correction (as its own docstring notes), which biases $L(r)-r$ downward as $r$ grows and approaches the size of the field — points near the boundary appear to have fewer neighbours than they really would in an infinite plane, simply because the count stops at the field's edge. This is why Section 6.2 and 6.3 stick to modest radii relative to the field size (`FIELD=100`, radii up to 25): a real analysis would either apply a proper edge correction or restrict measurements to points well inside a border guard band.

## 7 · Choosing a null distribution, spatial graphs, and what spatial analysis can (and cannot) tell you

CSR (complete spatial randomness) is only the simplest possible null distribution — fixing the number of objects and the region, then randomizing everything else. It is also the **most aggressive** choice: it destroys all spatial structure, which is sometimes exactly what you want and sometimes much too blunt an instrument. A short taxonomy, from gentlest to most aggressive:

- **Label randomization** — keep every object's position exactly fixed, shuffle only which objects are labelled "A" vs. "B." Tests whether *type* is related to position, without touching the spatial structure at all — the gentlest null, appropriate for "do A and B associate or segregate?"
- **Toroidal shift / translation** — preserve each pattern's own internal structure completely, and randomize only the relative alignment between two patterns (or images). Appropriate for "is pattern A's position related to pattern B's?"
- **Hard-core CSR** — complete spatial randomness, but with a enforced minimum spacing between objects (matching their physical size), avoiding CSR's tendency to falsely flag short-range "dispersion" among objects that simply cannot overlap.
- **Chunk/block shuffling** (the Costes approach from Section 4) — randomize at a block scale close to the point-spread function's width, respecting the fact that neighbouring pixels are not statistically independent at sub-resolution scales.
- **Plain CSR** — the default, and the most aggressive of the "legitimate" nulls: appropriate for one population in a roughly uniform region, but actively wrong when the true object density varies across the field (a density gradient will look like "clustering" against CSR even with no real clustering at all).
- **Pixel randomization** (shuffle every pixel independently) — destroys essentially everything, including real local continuity such as membranes; almost anything looks "significant" against it. Useful only as a sanity floor, never as a serious null.

In [ ]:
# ── 7.1 Delaunay and Voronoi: two more ways to turn coordinates into relationships ──
pts = simulate_point_pattern(n_points=40, size=FIELD, pattern="clustered", cluster_std=6.0, n_clusters=5, seed=SEED)
tri = Delaunay(pts)
vor = Voronoi(pts)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].triplot(pts[:, 1], pts[:, 0], tri.simplices, color="steelblue")
axes[0].scatter(pts[:, 1], pts[:, 0], color="crimson", zorder=3)
axes[0].set_title("Delaunay graph")
axes[1].scatter(pts[:, 1], pts[:, 0], color="crimson", zorder=3)
for ridge in vor.ridge_vertices:
    if -1 not in ridge:
        p1, p2 = vor.vertices[ridge[0]], vor.vertices[ridge[1]]
        axes[1].plot([p1[1], p2[1]], [p1[0], p2[0]], color="steelblue", linewidth=1)
axes[1].set_xlim(0, FIELD); axes[1].set_ylim(0, FIELD)
axes[1].set_title("Voronoi diagram")
plt.tight_layout()
plt.show()

🔍 **What just happened?** Nearest-neighbour search is not the only way to turn coordinates into relationships. A **Delaunay graph** connects points into a triangulation with no parameters to choose, creating a natural hierarchy of neighbours that can be biologically meaningful (e.g. roughly matching which cells actually touch) — but establishing it well on real, noisy segmentations can be technically demanding. A **Voronoi diagram** partitions space into one region per point (every location is assigned to its nearest point) — parameter-free and useful for area-based reasoning, but it can connect points that are biologically unrelated just because no other point sits between them over a long distance.

### 📘 Lecture notes — what spatial analysis is good for, and what it is not

Spatial analysis answers questions that genuinely require knowing *where* things are: how crowded is a region and does that crowding matter, who is near whom, whether an arrangement is organized or random, and whether two signals or object populations coincide more than chance predicts. It is **not** well suited to three other kinds of claims, and conflating them with what spatial statistics actually measure is a common and avoidable mistake. First, proximity is not interaction: showing that two proteins or two cells are close does not show that they interact — only a deliberate experimental design (knockouts, drug perturbation, or other targeted manipulation) can establish that. Second, spatial analysis shows a pattern exists but cannot, by itself, identify what *process* produced it — many different generative mechanisms can produce the same observed clustering. Third, as Section 6 demonstrated directly, answers from this kind of analysis are inherently scale-dependent: "these objects are clustered" is an incomplete claim until you also specify the radius at which that was measured.

## 📌 Key takeaways

- Every colocalization or spatial-pattern question reduces to the same two ingredients: a numerical measurement and a null distribution representing "no real relationship," and the right choice of each depends on the specific scientific question being asked.
- Pearson's correlation is the natural pixel-based colocalization measure when two channels occupy comparable areas; Manders' coefficients (the fraction of each channel's signal overlapping the other) suit the common case where they do not — but Manders' coefficients require a deliberately chosen threshold, since a naive near-zero threshold leaves them uninformatively close to 1 for almost any image.
- Costes' method addresses the threshold problem two ways: an automatic threshold derived from the regression between channels, and a block-shuffle randomization test for whether an observed correlation is larger than chance — and that test shows clearly that statistical significance and effect size are different questions, both worth reporting.
- Nearest-neighbour distances, the Clark-Evans index, and Ripley's L function all quantify point-pattern clustering or dispersion relative to a null (most simply, complete spatial randomness) — but only Ripley's L reveals the *scale* at which that clustering or dispersion happens, which a single global index like Clark-Evans cannot.
- CSR is the simplest null distribution but also one of the most aggressive — label randomization, toroidal shifts, hard-core CSR, and block shuffling are gentler, more targeted alternatives, and the right choice always follows from the specific question being asked, never from habit.
- Spatial and colocalization analysis can rigorously establish "more together than chance" or "more clustered than chance" — but cannot, on its own, establish molecular interaction, identify the underlying generative process, or answer a question without specifying the spatial scale at which it was asked.

## ✅ Checkpoint

1. Why are Manders' coefficients computed with a near-zero intensity threshold almost always close to 1, regardless of whether the two channels are truly associated?
   <details><summary>Answer</summary>With almost no threshold, nearly every nonzero pixel in each channel counts as "positive," so almost all of each channel's signal sits in pixels where the other channel is also technically "positive" — the measure is dominated by how permissive the threshold is, not by genuine spatial overlap.</details>
2. What specifically does Costes' automatic threshold method look for, to decide where to cut?
   <details><summary>Answer</summary>It scans a threshold down along the linear regression line relating the two channels, and stops at the highest threshold beyond which the population of pixels BELOW it shows zero or negative correlation — i.e. the point past which sub-threshold pixels look like uncorrelated background rather than real co-varying signal.</details>
3. Why does the Costes randomization test shuffle channel B in contiguous blocks rather than shuffling every pixel independently?
   <details><summary>Answer</summary>Because the point-spread function correlates neighbouring pixels (they are not statistically independent at sub-resolution scales); shuffling individual pixels would destroy that real local structure and make the null distribution too narrow, making almost any observed correlation look artificially significant. Blocks close to the PSF width preserve that local coupling while still destroying the large-scale alignment between the two channels.</details>
4. Two point patterns both have Clark-Evans index well below 1. What could Ripley's L tell you that Clark-Evans alone cannot?
   <details><summary>Answer</summary>Ripley's L as a function of radius reveals the spatial SCALE at which the clustering is strongest (where its curve peaks relative to the CSR envelope) — two patterns could both be "clustered" overall by Clark-Evans while clustering at very different characteristic distances, which only a radius-resolved measure like Ripley's L can distinguish.</details>

## 🧠 Questions & discussion

- Section 4 found that even a "near-independent" simulated scene (`association=0.0`) showed a small but statistically detectable Pearson correlation. List two plausible real-world sources (not simulation artifacts) of exactly this kind of small, genuine, non-biological baseline correlation between two microscopy channels.
- Section 7's null-distribution taxonomy ranked pixel randomization as "almost anything looks significant" and therefore not a serious null. Explain, in your own words, why a *destroying more structure* null distribution tends to make observed data look more significant, not less.
- Imagine two cell populations with a Clark-Evans cross-population index below 1 (apparent attraction). Sketch an experimental design (not a different statistic) that would let you distinguish "these cell types physically attract each other" from "both cell types are independently drawn toward some third factor, like a chemical gradient, that happens to co-localize them."

## 🚀 Bonus — going further

- Repeat Section 4's randomization test with `n_shuffles=1000` instead of `200`. Does the p-value for the near-independent scene change meaningfully, or does the extra precision mostly just narrow the null histogram's bin noise?
- Implement a simple **label-randomization** null (Section 7's gentlest option) for the cross-population question in Section 5: pool `pts_a` and `pts_b` together, repeatedly re-split the pooled points at random into two groups of the original sizes, and recompute the cross-population mean nearest-neighbour distance each time to build a null distribution — then compare the real `a_to_b.mean()` against it.
- Section 6.3 showed Ripley's L's peak location shifts with `cluster_std`. Sweep `cluster_std` over 4-5 values and plot peak radius vs. `cluster_std` directly — how close is the relationship to linear?

## 📝 Lab note

```text
Module 18 — Colocalization and spatial statistics
Data: SIMULATED two-channel scenes and point patterns, seed=18

Colocalization (high association / near-independent):
  Pearson r             [ ] / [ ]
  Manders M1 (naive)     [ ] / [ ]
  Manders M1 (Costes)    [ ] / [ ]
  Randomization p-value  [ ] / [ ]

Spatial statistics (random / clustered / dispersed):
  Clark-Evans index      [ ] / [ ] / [ ]
  Ripley's L peak radius (tight vs. loose clusters)  [ ] / [ ]

Notes:
```

## 📚 References

1. Manders, E. M. M., Verbeek, F. J., & Aten, J. A. (1993). Measurement of co-localization of objects in dual-colour confocal images. *Journal of Microscopy*.
2. Costes, S. V. et al. (2004). Automatic and quantitative measurement of protein-protein colocalization in live cells. *Biophysical Journal*.
3. Dunn, K. W., Kamocka, M. M., & McDonald, J. H. (2011). A practical guide to evaluating colocalization in biological microscopy. *American Journal of Physiology — Cell Physiology*.
4. Clark, P. J., & Evans, F. C. (1954). Distance to nearest neighbor as a measure of spatial relationships in populations. *Ecology*.
5. Ripley, B. D. (1977). Modelling spatial patterns. *Journal of the Royal Statistical Society, Series B*.
6. Diggle, P. J. (2013). *Statistical Analysis of Spatial and Spatio-Temporal Point Patterns* (3rd ed.). CRC Press.

## ➡️ Next

[**Module 19 — Attention, transformers, and diffusion**](19_attention_transformers_diffusion.ipynb): from classical statistical measurements, we turn to the architectures reshaping how images themselves are modelled — self-attention and vision transformers, and the diffusion process behind modern generative image models.